In [4]:
import cv2
import numpy as np
import pandas as pd
import joblib
from ultralytics import YOLO

# Load models
pose_model = YOLO('yolov8n-pose.pt')
classifier = joblib.load('model.pkl')

# Keypoints to drop (knees and ankles — same as training)
drop_kps = [13, 14, 15, 16]
drop_indices = []
for i in drop_kps:
    drop_indices += [i*3, i*3+1, i*3+2]

# Build feature column names matching what the model was trained on
feature_cols = [f'kp_{i}_{v}' for i in range(17) for v in ['x', 'y', 'conf']
                if i not in drop_kps]

print("Models loaded.")
print(f"Dropping feature indices: {drop_indices}")
print(f"Feature columns: {len(feature_cols)}")

Models loaded.
Dropping feature indices: [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
Feature columns: 39


In [5]:
cap = cv2.VideoCapture(0)
print("Press 'q' to quit")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, verbose=False, conf=0.7)
    annotated = results[0].plot()

    if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
        kps = results[0].keypoints.data[0].cpu().numpy().flatten()
        features = pd.DataFrame(np.delete(kps, drop_indices).reshape(1, -1), columns=feature_cols)
        prediction = classifier.predict(features)[0]

        label = "GOOD POSTURE" if prediction == 0 else "SLOUCH"
        color = (0, 255, 0) if prediction == 0 else (0, 0, 255)
    else:
        label = "No person detected"
        color = (255, 255, 255)

    cv2.putText(annotated, label, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

    cv2.imshow('Posture Monitor', annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit
